# Remaining E1: e5, Qwen3, LaBSE (10/25/40/50)

This runs the 12 missing lower-size configurations. bge-m3 is already done; do not rerun it here.

**Before running**

1. Runtime -> Change runtime type -> GPU (T4 is enough; A100 is faster).
2. Upload only these two files to `MyDrive/urdu_chunking/urdu_chunking_full/`:
   - `corpus_ur.jsonl`
   - `questions.csv`
3. The code is cloned from the public repository; you do not upload code.
4. Run the cells in order. The run is resumable: if Colab disconnects, rerun the clone/path and run cells.

Output is saved to `MyDrive/urdu_chunking/small_sizes_remaining/` and zipped for download at the end.


In [ ]:
!pip -q install sentence-transformers==6.2.0

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Make sure the two private data files exist in Drive.
# If they are somewhere else, this cell finds them and copies them to DEST.
# If they are not on Drive at all, it opens an upload dialog.
import os, shutil
from pathlib import Path

DEST = '/content/drive/MyDrive/urdu_chunking/urdu_chunking_full'
os.makedirs(DEST, exist_ok=True)
need = ['corpus_ur.jsonl', 'questions.csv']
missing = [n for n in need if not os.path.exists(os.path.join(DEST, n))]
print('missing:', missing)

if missing:
    for name in list(missing):
        hits = [p for p in Path('/content/drive/MyDrive').rglob(name) if p.is_file()]
        print(name, 'found at:', hits[:5])
        if hits:
            shutil.copy(hits[0], os.path.join(DEST, name))
    missing = [n for n in need if not os.path.exists(os.path.join(DEST, n))]
    if missing:
        print('Upload both files now (corpus_ur.jsonl and questions.csv):')
        from google.colab import files
        up = files.upload()
        for name in up:
            shutil.copy(name, os.path.join(DEST, name))

print('contents:', os.listdir(DEST))
for n in need:
    p = os.path.join(DEST, n)
    print(('OK  ' if os.path.exists(p) else 'MISS'), p,
          os.path.getsize(p) if os.path.exists(p) else '')

In [ ]:
import os, subprocess
REPO = 'https://github.com/devxyasir/urdu-chunking-strategy.git'
ART = '/content/urdu-chunking-artifact'
if not os.path.isdir(ART):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, ART], check=True)
else:
    subprocess.run(['git', '-C', ART, 'pull', '--ff-only'], check=True)
print('code at', ART)

In [ ]:
PKG = '/content/drive/MyDrive/urdu_chunking'   # <-- change if needed
DATA = f'{PKG}/urdu_chunking_full'
CODE = '/content/urdu-chunking-artifact/code'
OUT = f'{PKG}/small_sizes_remaining'
os.makedirs(OUT, exist_ok=True)
for p in [DATA + '/corpus_ur.jsonl', DATA + '/questions.csv', CODE + '/run_small_sizes.py',
          CODE + '/chunking.py', CODE + '/metrics.py']:
    print(('OK  ' if os.path.exists(p) else 'MISS'), p)
print('output:', OUT)

In [ ]:
import os, subprocess, sys

BATCH = 32   # lower to 16 if a T4 runs out of memory
cmd = [sys.executable, 'run_small_sizes.py',
       '--data-dir', DATA, '--out-dir', OUT,
       '--models', 'e5', 'qwen', 'labse',
       '--sizes', '10', '25', '40', '50',
       '--device', 'cuda', '--batch', str(BATCH)]
log_path = os.path.join(OUT, 'run.log')
print(' '.join(cmd))
env = dict(os.environ, PYTHONUNBUFFERED='1')
with open(log_path, 'a', encoding='utf-8') as log:
    p = subprocess.Popen(cmd, cwd=CODE, env=env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    rc = p.wait()
print('exit code:', rc, '| log:', log_path)
if rc != 0:
    raise SystemExit('The run stopped. Rerun this cell; completed runs are skipped.')

In [ ]:
import json, os
import pandas as pd

runs_path = os.path.join(OUT, 'runs.jsonl')
rows = []
if os.path.exists(runs_path):
    with open(runs_path, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
print('completed runs:', len(rows), 'of 12 expected')
if rows:
    df = pd.DataFrame([{**{'run_id': r['run_id'], 'model': r['model'].split('/')[-1],
                           'size': r['size'], 'chunks': r['chunks']}, **r['metrics']} for r in rows])
    df = df.sort_values(['model', 'size'])
    display(df[['model', 'size', 'chunks', 'bm500x', 'bm500x_lo', 'bm500x_hi', 'bm500', 'answer_split_rate']])
    print(df.groupby('model').size())
else:
    print('No completed runs yet.')

In [ ]:
import os, zipfile
from google.colab import files

archive = os.path.join(OUT, 'urdu_chunking_E1_remaining.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in ['runs.jsonl', 'summary.json', 'question_ids.csv', 'run.log']:
        p = os.path.join(OUT, name)
        if os.path.exists(p):
            z.write(p, arcname=name)
print('saved:', archive, os.path.getsize(archive), 'bytes')
files.download(archive)

## What to check before sending the file

- The table should show 12 runs: e5, Qwen3, and LaBSE at 10/25/40/50.
- If it shows fewer, rerun the run cell. Completed runs are skipped.
- Send `urdu_chunking_E1_remaining.zip` (or `runs.jsonl` and `summary.json`).
- Do not add `corpus_ur.jsonl` or `questions.csv` to the public repository.

Decision rule: if 40 or 25 beats 50 by at least 1 point with a paired CI excluding zero, the paper's
argmax changes. If not, 50 remains the best tested size for that encoder; if intervals overlap,
report a plateau rather than a sharp separation.
